In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data/ham10000")
meta = pd.read_csv(DATA / "HAM10000_metadata")   # no .csv ending in this version
paths = {p.stem: p for p in DATA.rglob("*.jpg")}
meta["path"] = meta["image_id"].map(paths)

print(meta.shape, "-", meta["path"].isna().sum(), "images missing")
meta["dx"].value_counts()

(10015, 9) - 0 images missing


dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

In [2]:
from sklearn.model_selection import train_test_split

SEED = 42

# keep only the four classes used for the benign vs. malignant task
meta = meta[meta["dx"].isin(["nv", "bkl", "mel", "bcc"])].copy()
meta["label"] = meta["dx"].isin(["mel", "bcc"]).astype(int)   # 1 = malignant

# split by lesion (not by image), so no lesion appears in two sets
lesions = meta.drop_duplicates("lesion_id")[["lesion_id", "label"]]
train_l, temp_l = train_test_split(lesions, test_size=0.30, stratify=lesions["label"], random_state=SEED)
val_l, test_l = train_test_split(temp_l, test_size=0.50, stratify=temp_l["label"], random_state=SEED)

split_of = {**{i: "train" for i in train_l["lesion_id"]},
            **{i: "val" for i in val_l["lesion_id"]},
            **{i: "test" for i in test_l["lesion_id"]}}
meta["split"] = meta["lesion_id"].map(split_of)

# sanity checks
assert meta.groupby("lesion_id")["label"].nunique().max() == 1   # one label per lesion
assert meta.groupby("lesion_id")["split"].nunique().max() == 1   # one split per lesion
print(meta.groupby(["split", "label"]).size().unstack())

# save the split so every experiment uses exactly the same one
meta[["image_id", "lesion_id", "dx", "label", "split"]].to_csv("../results/ham10000_split.csv", index=False)

label     0     1
split            
test   1178   264
train  5449  1120
val    1177   243


In [3]:
import sys
sys.path.append("..")          # so the notebook can import from src/

import pandas as pd
from torch.utils.data import DataLoader
from src.data import load_ham10000, SkinDataset, get_transforms
from src.train import set_seed, get_device, build_model, train

SEED = 42
set_seed(SEED)
device = get_device()
print("Using:", device)

df = load_ham10000("../data/ham10000", "../results/ham10000_split.csv")

# small subset, just to check that the whole pipeline works
train_df = df[df["split"] == "train"].sample(800, random_state=SEED)
val_df = df[df["split"] == "val"].sample(200, random_state=SEED)

train_loader = DataLoader(SkinDataset(train_df, get_transforms(augment=False)), batch_size=32, shuffle=True)
val_loader = DataLoader(SkinDataset(val_df, get_transforms(augment=False)), batch_size=32)

model = build_model().to(device)
history = train(model, train_loader, val_loader, device, epochs=2)

pd.DataFrame(history).to_csv("../results/poc_baseline_history.csv", index=False)

ModuleNotFoundError: No module named 'torch'

In [5]:
%pip install torch torchvision


  Using cached torchvision-0.29.2-py3-none-win_amd64.whl.metadata (5.7 kB)
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/126.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/126.7 MB 1.8 MB/s eta 0:01:12
   ---------------------------------------- 1.0/126.7 MB 1.7 MB/s eta 0:01:13
   ---------------------------------------- 1.3/126.7 MB 1.9 MB/s eta 0:01:07
    --------------------------------------- 1.8/126.7 MB 1.8 MB/s eta 0:01:11
    --------------------------------------- 2.4/126.7 MB 1.9 MB/s eta 0:01:06
    --------------------------------------- 2.9/126.7 MB 2.0 MB/s eta 0:01:02
   - -------------------------------------- 3.4/126.7 MB 2.1 MB/s eta 0:00:58
   - -------------------------------------- 3.9/126.7 MB 2.2 MB/s eta 0:00:56
   - -------------------------------------- 5.0/126.7 MB 2.4 MB/s eta 0:00:51
   - -------------------------------------- 5.8/126.7 MB 2.6 MB/s eta 0:00:48
   

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
